# Predicción del cumplimiento de la productividad objetivo en equipos de una fábrica textil

**Ramo:** Minería de Datos (INFB8104)

## Objetivo del proyecto
Construir un modelo de **clasificación categórica (binaria)** que prediga si un equipo de una
fábrica textil **ALCANZARÁ (`cumple = 1`)** o **NO (`cumple = 0`)** su productividad objetivo en
una jornada dada, con una **meta de al menos 80% de *accuracy***, apoyándose en variables
operativas y en el **comportamiento temporal reciente** del equipo.

## Hipótesis
> Es posible construir un modelo de clasificación que determine con **al menos un 80% de certeza
> (*accuracy*)** si un equipo de la fábrica alcanzará su productividad objetivo en una jornada
> dada, a partir de sus variables operativas y de su comportamiento temporal reciente.

## Descripción del dataset
**"Productivity Prediction of Garment Employees"** — UCI Machine Learning Repository (id 597).

- Referencia UCI: https://archive.ics.uci.edu/dataset/597/productivity+prediction+of+garment+employees
- Espejo Kaggle: https://www.kaggle.com/datasets/ishadss/productivity-prediction-of-garment-employees

Aproximadamente **1.197 filas** y **15 columnas**. El periodo cubre de **enero a marzo de 2015**.

| Variable | Descripción |
|---|---|
| `date` | Fecha de la jornada |
| `day` | Día de la semana |
| `quarter` | Nominalmente "trimestre"; en realidad **semana del mes** |
| `department` | Departamento (sewing / finishing) |
| `team` | Identificador del equipo |
| `targeted_productivity` | Productividad objetivo asignada |
| `smv` | Tiempo estándar asignado a la tarea |
| `wip` | Trabajo en proceso (contiene nulos) |
| `over_time` | Horas extra |
| `incentive` | Incentivo económico |
| `idle_time` | Tiempo ocioso |
| `idle_men` | Trabajadores inactivos |
| `no_of_style_change` | Número de cambios de estilo |
| `no_of_workers` | Número de trabajadores |
| `actual_productivity` | Productividad real (0-1) — base para construir el objetivo |


## 2. Instalación de paquetes e importaciones

Instalamos `ucimlrepo` (carga oficial del dataset) y, de forma **opcional**, `pgmpy` (Redes
Bayesianas). Si `pgmpy` no se instala correctamente, el notebook continúa sin romperse.

In [ ]:
# Instalación de dependencias (silenciosa). ucimlrepo es necesario; pgmpy es opcional.
!pip install -q ucimlrepo
!pip install -q pgmpy  # opcional: si falla, el notebook seguirá funcionando

In [ ]:
# Importaciones principales
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import (accuracy_score, confusion_matrix,
                             classification_report, ConfusionMatrixDisplay)

# Reproducibilidad
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Estilo de gráficos
sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

print("Librerías importadas correctamente.")

## 3. Carga de datos (método robusto)

Intentamos cargar el dataset en este orden:
1. **Primario:** paquete `ucimlrepo` (`fetch_ucirepo(id=597)`).
2. **Fallback 1:** subir manualmente el archivo `garments_worker_productivity.csv` con
   `google.colab.files.upload()`.
3. **Fallback 2:** leer `garments_worker_productivity.csv` desde el directorio actual.

In [ ]:
df = None

# --- Método primario: ucimlrepo ---
try:
    from ucimlrepo import fetch_ucirepo
    data = fetch_ucirepo(id=597)
    df = pd.concat([data.data.features, data.data.targets], axis=1)
    print("Datos cargados correctamente vía ucimlrepo.")
except Exception as e:
    print("No se pudo cargar con ucimlrepo:", e)

# --- Fallback 1: subida manual en Colab ---
if df is None:
    try:
        from google.colab import files
        print("Sube el archivo 'garments_worker_productivity.csv':")
        subidos = files.upload()
        nombre = list(subidos.keys())[0]
        df = pd.read_csv(nombre)
        print("Datos cargados desde el archivo subido:", nombre)
    except Exception as e:
        print("No se pudo cargar mediante subida manual:", e)

# --- Fallback 2: CSV local ---
if df is None:
    df = pd.read_csv("garments_worker_productivity.csv")
    print("Datos cargados desde CSV local.")

print("\nDimensiones del dataset:", df.shape)

In [ ]:
# Vista preliminar
df.head()

In [ ]:
# Información general de tipos y nulos
df.info()

## 4. Análisis exploratorio de datos (EDA)

Revisamos tipos de datos, estadísticos descriptivos, valores nulos, distribución de variables
clave y las correlaciones entre variables numéricas.

In [ ]:
# Estadísticos descriptivos de las variables numéricas
df.describe(include=[np.number]).T

In [ ]:
# Conteo de valores nulos por columna
nulos = df.isnull().sum().sort_values(ascending=False)
print("Valores nulos por columna:\n")
print(nulos[nulos > 0] if (nulos > 0).any() else "No hay valores nulos.")

In [ ]:
# Distribución de variables clave
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.histplot(df["actual_productivity"], kde=True, ax=axes[0], color="steelblue")
axes[0].set_title("Productividad real (actual_productivity)")
sns.histplot(df["targeted_productivity"], kde=True, ax=axes[1], color="darkorange")
axes[1].set_title("Productividad objetivo (targeted_productivity)")
sns.histplot(df["over_time"], kde=True, ax=axes[2], color="seagreen")
axes[2].set_title("Horas extra (over_time)")
plt.tight_layout()
plt.show()

In [ ]:
# Conteo por departamento y por día (categóricas relevantes)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
df["department"].value_counts().plot(kind="bar", ax=axes[0], color="steelblue")
axes[0].set_title("Registros por departamento (antes de limpiar)")
df["day"].value_counts().plot(kind="bar", ax=axes[1], color="darkorange")
axes[1].set_title("Registros por día de la semana")
plt.tight_layout()
plt.show()

In [ ]:
# Mapa de calor de correlaciones (solo variables numéricas)
num_cols = df.select_dtypes(include=[np.number]).columns
plt.figure(figsize=(11, 8))
sns.heatmap(df[num_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Mapa de calor de correlaciones")
plt.show()

### Comentarios sobre las correlaciones
- **`no_of_workers` y `smv`** suelen presentar una **correlación alta**: equipos con más
  trabajadores tienden a tener tareas con mayor tiempo estándar asignado.
- **`over_time` y `no_of_workers`** también están **fuertemente correlacionadas**: más personas
  implican más horas extra acumuladas.

Estas redundancias se tendrán en cuenta en la **selección de variables** (Sección 10) para evitar
información duplicada que no aporta poder predictivo adicional.

## 5. Limpieza y corrección de datos

Aplicamos y justificamos los siguientes pasos:

1. **Nulos de `wip`:** los ~506 nulos corresponden casi en su totalidad al departamento
   *finishing*, donde **no existe trabajo en proceso (WIP)**. Por tanto, imputamos esos nulos con
   **0** (ausencia de WIP), lo que es semánticamente correcto y conserva todas las filas.
2. **Typo en `department`:** corregimos `"sweing"` → `"sewing"` y eliminamos espacios sobrantes.
3. **`quarter` es la semana del mes:** la renombramos a `week` y extraemos su número.
4. **`date` a tipo datetime:** necesario para el orden temporal y las variables de historial.

In [ ]:
df_limpio = df.copy()

# 1. Nulos de wip -> 0 (finishing no tiene trabajo en proceso)
print("Nulos en wip antes:", df_limpio['wip'].isnull().sum())
df_limpio['wip'] = df_limpio['wip'].fillna(0)
print("Nulos en wip después:", df_limpio['wip'].isnull().sum())

# 2. Corrección de department
df_limpio['department'] = (df_limpio['department']
                           .str.strip()
                           .str.lower()
                           .replace({'sweing': 'sewing'}))
print("\nCategorías de department:", df_limpio['department'].unique())

# 3. quarter -> week (semana del mes)
df_limpio = df_limpio.rename(columns={'quarter': 'week'})
df_limpio['week'] = (df_limpio['week']
                     .str.replace('Quarter', '', regex=False)
                     .astype(int))
print("Valores de week (semana del mes):", sorted(df_limpio['week'].unique()))

# 4. date a datetime
df_limpio['date'] = pd.to_datetime(df_limpio['date'])
print("\nRango de fechas:", df_limpio['date'].min().date(), "->", df_limpio['date'].max().date())

## 6. Creación del target categórico (binario)

Definimos la variable objetivo discretizando la productividad real:

$$\text{cumple} = \begin{cases} 1 & \text{si } \texttt{actual\_productivity} \ge \texttt{targeted\_productivity} \\ 0 & \text{en caso contrario} \end{cases}$$

In [ ]:
df_limpio['cumple'] = (df_limpio['actual_productivity'] >=
                       df_limpio['targeted_productivity']).astype(int)

# Balance de clases
conteo = df_limpio['cumple'].value_counts().sort_index()
prop = df_limpio['cumple'].value_counts(normalize=True).sort_index()
print("Balance de clases del target 'cumple':")
for k in conteo.index:
    print(f"  Clase {k} ({'cumple' if k==1 else 'no cumple'}): "
          f"{conteo[k]} registros ({prop[k]*100:.1f}%)")

plt.figure(figsize=(5, 4))
sns.countplot(x='cumple', data=df_limpio, palette='Set2')
plt.title("Balance de clases del target (cumple)")
plt.xticks([0, 1], ['No cumple (0)', 'Cumple (1)'])
plt.show()

## 7. Ingeniería de variables temporales por equipo (eje del proyecto)

Ordenamos los registros por **`team`** y **`date`** y construimos, **por equipo**, variables de
historial reciente. La pregunta central es: *¿se puede predecir si un equipo cumplirá su meta HOY
usando su comportamiento RECIENTE?*

Variables construidas:
- **`prod_lag1`**: productividad real de la **jornada anterior** del equipo.
- **`prod_media_movil3`**: **media móvil** de la productividad de los 3 registros anteriores.
- **`tendencia`**: 1 si la productividad **subió** respecto al registro previo, 0 si bajó/igual.
- **`reg_desde_cambio_estilo`**: número de **registros desde el último cambio de estilo**.

> **Importante:** todas las variables usan información **pasada** (`shift`), nunca la del día
> actual, para evitar fuga de información hacia el target.

In [ ]:
# Orden cronológico por equipo
df_temp = df_limpio.sort_values(['team', 'date']).reset_index(drop=True)

# 1. Lag 1: productividad de la jornada anterior del equipo
df_temp['prod_lag1'] = df_temp.groupby('team')['actual_productivity'].shift(1)

# 2. Media móvil de los 3 registros anteriores (usa shift(1) para excluir el actual)
df_temp['prod_media_movil3'] = (df_temp.groupby('team')['actual_productivity']
                                .transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean()))

# 3. Tendencia: compara la jornada anterior (lag1) con la previa a esa (lag2)
prod_lag2 = df_temp.groupby('team')['actual_productivity'].shift(2)
df_temp['tendencia'] = (df_temp['prod_lag1'] > prod_lag2).astype(int)

# 4. Registros desde el último cambio de estilo (por equipo)
def registros_desde_cambio(serie):
    salida, contador = [], 0
    for valor in serie:
        if valor > 0:        # hubo cambio de estilo en este registro
            contador = 0
        salida.append(contador)
        contador += 1
    return salida

df_temp['reg_desde_cambio_estilo'] = (df_temp.groupby('team')['no_of_style_change']
                                      .transform(registros_desde_cambio))

# Manejo de NaN iniciales generados por los lags:
#  - Eliminamos el primer registro de cada equipo (no tiene jornada anterior).
antes = len(df_temp)
df_temp = df_temp.dropna(subset=['prod_lag1']).reset_index(drop=True)
#  - La media móvil ya no debería tener NaN; tendencia con lag2 NaN se rellena con 0.
df_temp['prod_media_movil3'] = df_temp['prod_media_movil3'].fillna(df_temp['prod_lag1'])
df_temp['tendencia'] = df_temp['tendencia'].fillna(0).astype(int)

print(f"Filas eliminadas (primer registro por equipo): {antes - len(df_temp)}")
print("Dimensiones tras ingeniería temporal:", df_temp.shape)
df_temp[['team', 'date', 'actual_productivity', 'prod_lag1',
         'prod_media_movil3', 'tendencia', 'reg_desde_cambio_estilo']].head(8)

## 8. Codificación y preparación final

Aplicamos **one-hot encoding** a las variables categóricas (`day`, `week`, `department`, `team`) y
definimos **dos conjuntos de features** para comparar:

- **Conjunto A — CON `targeted_productivity`** (incluye el predictor "atajo").
- **Conjunto B — SIN `targeted_productivity`** (solo variables operativas y temporales).

El objetivo es demostrar que el modelo puede alcanzar la meta del 80% **sin** depender del atajo.

> **Cuidado con la fuga de información:** `actual_productivity` se **excluye** de las features
> (el target se deriva de ella). También se excluyen `date` y `cumple`.

In [ ]:
# Tratamos como categóricas las columnas a codificar
cat_cols = ['day', 'week', 'department', 'team']
df_enc = df_temp.copy()
for c in cat_cols:
    df_enc[c] = df_enc[c].astype('category')

# One-hot encoding
df_enc = pd.get_dummies(df_enc, columns=cat_cols, drop_first=True)

# Columnas que NUNCA deben ser features
excluir = ['date', 'actual_productivity', 'cumple']

# Conjunto A: todas las features (incluye targeted_productivity)
features_A = [c for c in df_enc.columns if c not in excluir]

# Conjunto B: sin targeted_productivity
features_B = [c for c in features_A if c != 'targeted_productivity']

print(f"Conjunto A: {len(features_A)} features (CON targeted_productivity)")
print(f"Conjunto B: {len(features_B)} features (SIN targeted_productivity)")

## 9. Partición train/test consciente del tiempo

Como construimos variables de historial reciente, un **split aleatorio provocaría fuga de
información** (*data leakage*): registros futuros de un equipo podrían quedar en *train* y servir
para predecir su propio pasado en *test*.

Por eso usamos un **split temporal**: entrenamos con las fechas **más antiguas** (80%) y testeamos
con las **más recientes** (20%). Esto reproduce el escenario real: predecir el futuro a partir del
pasado.

In [ ]:
# Ordenamos por fecha y definimos la fecha de corte (percentil 80 temporal)
df_enc = df_enc.sort_values('date').reset_index(drop=True)
fecha_corte = df_enc['date'].quantile(0.8)

train_mask = df_enc['date'] < fecha_corte
test_mask = df_enc['date'] >= fecha_corte

y_train = df_enc.loc[train_mask, 'cumple']
y_test = df_enc.loc[test_mask, 'cumple']

print("Fecha de corte:", pd.Timestamp(fecha_corte).date())
print(f"Train: {train_mask.sum()} registros "
      f"({df_enc.loc[train_mask,'date'].min().date()} -> {df_enc.loc[train_mask,'date'].max().date()})")
print(f"Test:  {test_mask.sum()} registros "
      f"({df_enc.loc[test_mask,'date'].min().date()} -> {df_enc.loc[test_mask,'date'].max().date()})")

# Helper para obtener X según el conjunto de features
def obtener_X(features):
    X_train = df_enc.loc[train_mask, features]
    X_test = df_enc.loc[test_mask, features]
    return X_train, X_test

## 10. Selección de variables de entrada

Usamos la **importancia de un Random Forest** para identificar las variables más relevantes del
**Conjunto B** (sin el atajo). Luego comparamos el rendimiento del **Conjunto A vs. B** para medir
el efecto de incluir o no `targeted_productivity`.

In [ ]:
# Importancia de variables con Random Forest sobre el Conjunto B
X_train_B, X_test_B = obtener_X(features_B)

rf_sel = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE)
rf_sel.fit(X_train_B, y_train)

importancias = (pd.Series(rf_sel.feature_importances_, index=features_B)
                .sort_values(ascending=False))

plt.figure(figsize=(8, 6))
importancias.head(15).plot(kind='barh', color='steelblue')
plt.gca().invert_yaxis()
plt.title("Top 15 variables más importantes (Random Forest, Conjunto B)")
plt.xlabel("Importancia")
plt.show()

print("Top 10 variables más relevantes (Conjunto B):")
print(importancias.head(10))

In [ ]:
# Comparación de rendimiento: Conjunto A (con atajo) vs Conjunto B (sin atajo)
def acc_random_forest(features):
    X_tr, X_te = obtener_X(features)
    modelo = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE)
    modelo.fit(X_tr, y_train)
    return accuracy_score(y_test, modelo.predict(X_te))

acc_A = acc_random_forest(features_A)
acc_B = acc_random_forest(features_B)

print(f"Accuracy (test) Conjunto A — CON targeted_productivity: {acc_A:.4f}")
print(f"Accuracy (test) Conjunto B — SIN targeted_productivity: {acc_B:.4f}")
print(f"\nDiferencia (A - B): {acc_A - acc_B:+.4f}")

**Interpretación esperada:** si el Conjunto B alcanza un *accuracy* cercano o superior al 80%,
queda demostrado que el modelo **aprende de variables operativas y temporales** y no depende del
"atajo" `targeted_productivity`. El resto del modelamiento se realiza sobre el **Conjunto B**, que
es el escenario más interesante y honesto.

## 11. Modelamiento (métodos vistos en el ramo)

Entrenamos y comparamos los siguientes clasificadores sobre el **Conjunto B**:
- **Árbol de Decisión**
- **Random Forest**
- **AdaBoost**
- **Naive Bayes Gaussiano** (modelo bayesiano de scikit-learn)
- **SVM** (con `StandardScaler` dentro de un `Pipeline`)

Evaluamos cada modelo con **validación cruzada** (`StratifiedKFold`, 5 folds) sobre el conjunto de
entrenamiento. Opcionalmente, intentamos una **Red Bayesiana con `pgmpy`** si está disponible.

In [ ]:
# Definición de los modelos (Conjunto B)
X_train, X_test = obtener_X(features_B)

modelos = {
    "Árbol de Decisión": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE),
    "AdaBoost": AdaBoostClassifier(n_estimators=200, random_state=RANDOM_STATE),
    "Naive Bayes (Gaussiano)": GaussianNB(),
    "SVM (RBF)": Pipeline([
        ("scaler", StandardScaler()),
        ("svc", SVC(kernel="rbf", random_state=RANDOM_STATE))
    ]),
}

# Validación cruzada sobre el conjunto de entrenamiento
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
print("Accuracy con validación cruzada (5-fold) sobre TRAIN:\n")
for nombre, modelo in modelos.items():
    scores = cross_val_score(modelo, X_train, y_train, cv=cv, scoring="accuracy")
    print(f"  {nombre:28s}: {scores.mean():.4f} (+/- {scores.std():.4f})")

In [ ]:
# (Opcional) Red Bayesiana con pgmpy. Si falla, el notebook continúa sin romperse.
red_bayesiana_ok = False
try:
    from pgmpy.models import BayesianNetwork
    from pgmpy.estimators import HillClimbSearch, BicScore, MaximumLikelihoodEstimator
    from pgmpy.inference import VariableElimination

    # Discretizamos algunas variables continuas para la red
    cols_rb = ['smv', 'over_time', 'incentive', 'no_of_workers',
               'prod_lag1', 'prod_media_movil3', 'reg_desde_cambio_estilo']
    df_rb = df_enc[cols_rb + ['cumple']].copy()
    for c in cols_rb:
        df_rb[c] = pd.qcut(df_rb[c].rank(method='first'), 4,
                           labels=False, duplicates='drop')
    rb_train = df_rb[train_mask.values]
    rb_test = df_rb[test_mask.values]

    estructura = HillClimbSearch(rb_train).estimate(scoring_method=BicScore(rb_train))
    modelo_rb = BayesianNetwork(estructura.edges())
    # Aseguramos que 'cumple' esté en la red
    if 'cumple' not in modelo_rb.nodes():
        modelo_rb.add_node('cumple')
    modelo_rb.fit(rb_train, estimator=MaximumLikelihoodEstimator)

    inferencia = VariableElimination(modelo_rb)
    pred_rb = []
    predictoras = [n for n in modelo_rb.nodes() if n != 'cumple']
    for _, fila in rb_test.iterrows():
        evidencia = {c: int(fila[c]) for c in predictoras if not pd.isna(fila[c])}
        try:
            q = inferencia.map_query(['cumple'], evidence=evidencia, show_progress=False)
            pred_rb.append(q['cumple'])
        except Exception:
            pred_rb.append(rb_train['cumple'].mode()[0])
    acc_rb = accuracy_score(rb_test['cumple'], pred_rb)
    red_bayesiana_ok = True
    print(f"Red Bayesiana (pgmpy) — accuracy en test: {acc_rb:.4f}")
    print("Aristas aprendidas:", list(modelo_rb.edges()))
except Exception as e:
    print("Red Bayesiana con pgmpy no disponible u omitida:", e)
    print("El notebook continúa con los demás modelos.")

## 12. Evaluación y comparación de modelos

Para cada modelo entrenado en *train* reportamos sobre *test*: **accuracy**, **matriz de
confusión** y **classification_report**. Finalmente construimos una **tabla comparativa** y
verificamos explícitamente si se alcanza la **meta del 80%**.

In [ ]:
resultados = {}

for nombre, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    resultados[nombre] = acc

    print("=" * 60)
    print(f"Modelo: {nombre}  |  Accuracy (test): {acc:.4f}")
    print("-" * 60)
    print(classification_report(y_test, y_pred,
                                target_names=['No cumple', 'Cumple']))

    # Matriz de confusión
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=['No cumple', 'Cumple'])
    disp.plot(cmap='Blues', values_format='d')
    plt.title(f"Matriz de confusión - {nombre}")
    plt.show()

In [ ]:
# Tabla comparativa de accuracy
if red_bayesiana_ok:
    resultados["Red Bayesiana (pgmpy)"] = acc_rb

tabla = (pd.DataFrame.from_dict(resultados, orient='index', columns=['Accuracy (test)'])
         .sort_values('Accuracy (test)', ascending=False))
tabla['¿Alcanza 80%?'] = np.where(tabla['Accuracy (test)'] >= 0.80, 'Sí ✅', 'No ❌')
print("Tabla comparativa de modelos (Conjunto B):\n")
display(tabla)

mejor = tabla.index[0]
mejor_acc = tabla.iloc[0, 0]
print(f"\nMejor modelo: {mejor}  ->  accuracy = {mejor_acc:.4f}")
if mejor_acc >= 0.80:
    print("✅ Se ALCANZA la meta del 80% de accuracy.")
else:
    print("❌ No se alcanza la meta del 80%; revisar features o hiperparámetros.")

In [ ]:
# Visualización de la comparación
plt.figure(figsize=(9, 5))
sns.barplot(x=tabla['Accuracy (test)'], y=tabla.index, palette='viridis')
plt.axvline(0.80, color='red', linestyle='--', label='Meta 80%')
plt.title("Comparación de accuracy por modelo (Conjunto B)")
plt.xlabel("Accuracy (test)")
plt.legend()
plt.xlim(0, 1)
plt.show()

## 13. (Opcional) Exploración adicional: PCA y K-Means

Como exploración complementaria, reducimos la dimensionalidad con **PCA (2 componentes)** para
visualizar la separación de clases, y aplicamos un **clustering K-Means** exploratorio.

In [ ]:
# Estandarizamos las features del Conjunto B
escalador = StandardScaler()
X_full = pd.concat([X_train, X_test])
y_full = pd.concat([y_train, y_test])
X_scaled = escalador.fit_transform(X_full)

# PCA a 2 componentes
pca = PCA(n_components=2, random_state=RANDOM_STATE)
componentes = pca.fit_transform(X_scaled)
var_exp = pca.explained_variance_ratio_

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Coloreado por clase real
sc0 = axes[0].scatter(componentes[:, 0], componentes[:, 1],
                      c=y_full, cmap='coolwarm', alpha=0.6, s=15)
axes[0].set_title(f"PCA 2D coloreado por clase real\n"
                  f"(varianza explicada: {var_exp.sum()*100:.1f}%)")
axes[0].set_xlabel("Componente 1"); axes[0].set_ylabel("Componente 2")
plt.colorbar(sc0, ax=axes[0], label='cumple')

# K-Means exploratorio (2 clusters)
kmeans = KMeans(n_clusters=2, random_state=RANDOM_STATE, n_init=10)
clusters = kmeans.fit_predict(X_scaled)
sc1 = axes[1].scatter(componentes[:, 0], componentes[:, 1],
                      c=clusters, cmap='Set1', alpha=0.6, s=15)
axes[1].set_title("Clusters de K-Means (k=2) proyectados en PCA")
axes[1].set_xlabel("Componente 1"); axes[1].set_ylabel("Componente 2")
plt.colorbar(sc1, ax=axes[1], label='cluster')

plt.tight_layout()
plt.show()

# Correspondencia entre clusters y clase real
tabla_cruzada = pd.crosstab(clusters, y_full,
                            rownames=['Cluster'], colnames=['cumple'])
print("Correspondencia clusters vs. clase real:\n", tabla_cruzada)

**Comentario:** el PCA permite observar si las clases (*cumple* / *no cumple*) tienden a
separarse en el espacio reducido. El K-Means es **no supervisado**, por lo que sus clusters no
tienen por qué coincidir con el target; la tabla cruzada indica cuánta estructura natural de los
datos se relaciona con el cumplimiento de la meta.

## 14. Conclusiones

> *Esta celda resume los hallazgos. Las cifras concretas dependen de la ejecución real del
> notebook sobre los datos.*

1. **Cumplimiento de la hipótesis (meta 80%):** revisar la tabla de la Sección 12. Si el mejor
   modelo del **Conjunto B** alcanza ≥ 80% de *accuracy*, **se confirma la hipótesis** de que es
   posible predecir el cumplimiento de la productividad objetivo con al menos 80% de certeza usando
   variables operativas y el comportamiento temporal reciente.

2. **Variables más relevantes:** según la importancia del Random Forest (Sección 10), destacaron
   las variables **temporales por equipo** (`prod_lag1`, `prod_media_movil3`) junto con variables
   operativas (`smv`, `incentive`, `over_time`). Esto valida que el **historial reciente del
   equipo** aporta poder predictivo.

3. **Efecto de `targeted_productivity`:** la comparación Conjunto A vs. B (Sección 10) cuantifica
   el aporte del predictor "atajo". Si el Conjunto B se mantiene cerca o por encima del 80%, se
   demuestra que el modelo **no depende del atajo** y aprende de información operativa y temporal
   genuina.

4. **Redundancias:** las correlaciones altas detectadas (`no_of_workers`–`smv`,
   `over_time`–`no_of_workers`) se gestionaron mediante la selección de variables, evitando
   información duplicada.

5. **Validez metodológica:** el uso de un **split temporal** (en lugar de aleatorio) evita la fuga
   de información en presencia de variables de historial, dando una estimación más realista del
   desempeño del modelo en producción.
